In [ ]:
from __future__ import print_function
import datetime
import os.path
from googleapiclient.discovery import build
from google_auth_oauthlib.flow import InstalledAppFlow
from google.auth.transport.requests import Request
import pickle

# Define the scope for the Google Calendar API
SCOPES = ['https://www.googleapis.com/auth/calendar.readonly']

def get_calendar_events(start_date, end_date):
    """Retrieve events from the Google Calendar within a specific date range."""
    creds = None
    # Check if token.pickle exists (contains user credentials)
    if os.path.exists('token.pickle'):
        with open('token.pickle', 'rb') as token:
            creds = pickle.load(token)

    # If no valid credentials are available, ask the user to log in
    if not creds or not creds.valid:
        if creds and creds.expired and creds.refresh_token:
            creds.refresh(Request())
        else:
            flow = InstalledAppFlow.from_client_secrets_file(
                'credentials.json', SCOPES)
            creds = flow.run_local_server(port=0)
        # Save the credentials for future use
        with open('token.pickle', 'wb') as token:
            pickle.dump(creds, token)

    service = build('calendar', 'v3', credentials=creds)

    # Convert dates to ISO format
    start_date = start_date.isoformat() + 'Z'  # 'Z' indicates UTC time
    end_date = end_date.isoformat() + 'Z'

    # Call the Calendar API to fetch events
    events_result = service.events().list(
        calendarId='primary',
        timeMin=start_date,
        timeMax=end_date,
        singleEvents=True,
        orderBy='startTime'
    ).execute()

    events = events_result.get('items', [])

    if not events:
        print('No events found in the specified date range.')
        return []

    print('Events in the specified date range:')
    for event in events:
        start = event['start'].get('dateTime', event['start'].get('date'))
        print(f"{event['summary']} ({start})")

    return events

if __name__ == '__main__':
    # Specify the date range for querying events
    start_date = datetime.datetime(2024, 1, 1)
    end_date = datetime.datetime(2024, 1, 31)

    events = get_calendar_events(start_date, end_date)

    # Output details of each event
    for event in events:
        print("\nEvent Details:")
        print(f"Summary: {event.get('summary', 'No Title')}")
        print(f"Start: {event['start'].get('dateTime', event['start'].get('date'))}")
        print(f"End: {event['end'].get('dateTime', event['end'].get('date'))}")
        print(f"Location: {event.get('location', 'No Location')}")
        print(f"Description: {event.get('description', 'No Description')}")
